In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from time import sleep
import base64

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 6)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("No GitHub tokens found in All_Tokens.env")

token_index = 0
def get_headers():
    global token_index
    token = tokens[token_index]
    token_index = (token_index + 1) % len(tokens)
    print(f"🔁 Using token #{token_index + 1}")
    return {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "android-repo-crawler/1.0"
    }

# === File paths ===
input_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step2_verified_output.csv"
output_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step3_android_detection_output.csv"

# === Read data ===
df = pd.read_csv(input_path)

android_in_readme = []
confidence = []

def search_android_in_files(repo):
    # Check README first
    readme_url = f"https://api.github.com/repos/{repo}/readme"
    response = requests.get(readme_url, headers=get_headers())
    if response.status_code == 200:
        content = response.json().get("content", "")
        decoded = base64.b64decode(content).decode("utf-8", errors="ignore").lower()
        if "android" in decoded:
            return True

    # Check other contents for .md or .rst
    contents_url = f"https://api.github.com/repos/{repo}/contents"
    response = requests.get(contents_url, headers=get_headers())
    if response.status_code == 200:
        for item in response.json():
            if item["name"].lower().endswith(('.md', '.markdown', '.rst')):
                file_url = item["download_url"]
                if file_url:
                    text = requests.get(file_url, headers=get_headers()).text.lower()
                    if "android" in text:
                        return True

    # Optionally check GitHub Wiki main page (heuristic, not via API)
    wiki_url = f"https://raw.githubusercontent.com/wiki/{repo}/Home.md"
    response = requests.get(wiki_url, headers={"User-Agent": "android-repo-crawler/1.0"})
    if response.status_code == 200 and "android" in response.text.lower():
        return True

    return False

# === Process each repo ===
for i, row in df.iterrows():
    if row["status"] != "pass":
        android_in_readme.append("N/A")
        confidence.append("N/A")
        continue

    found = False
    name = str(row.get("name", "")).lower()
    desc = str(row.get("description", "")).lower()
    topics = str(row.get("topics", "")).lower()

    if "android" in name or "android" in desc or "android" in topics:
        found = True

    in_readme_or_alt = search_android_in_files(row["full_name"])

    android_in_readme.append("yes" if in_readme_or_alt else "no")
    if in_readme_or_alt and not found:
        confidence.append("low")
    elif found:
        confidence.append("high")
    else:
        confidence.append("none")

    if i % 100 == 0:
        print(f"🔍 Checked {i+1} repos...")

# === Save output ===
df["android_in_readme"] = android_in_readme
df["confidence"] = confidence
df.to_csv(output_path, index=False)
print(f"✅ Step 3 complete. Saved to: {output_path}")
